In [6]:

!pip install hdbscan



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
# -----------------------------
# 1. LOAD DATA
# -----------------------------
df = pd.read_csv("makerere_Cafeteria_synthetic.csv")

cols = ['Portions_Sold', 'Waste_Portions', 'Gross_Profit_UGX']

for col in cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Fill missing values after conversion
df[cols] = df[cols].fillna(0)

# -----------------------------
# 2. CREATE DERIVED FEATURES
# -----------------------------
df['Waste_Rate'] = df['Waste_Portions'] / df['Portions_Sold'].replace(0, np.nan)
df['Profit_Margin'] = df['Gross_Profit_UGX'] / df['Portions_Sold'].replace(0, np.nan)

df[['Waste_Rate', 'Profit_Margin']] = df[['Waste_Rate', 'Profit_Margin']].fillna(0)

# -----------------------------
# 3. SELECT FEATURES
# -----------------------------
features = df[[
    'Portions_Sold',
    'Waste_Portions',
    'Gross_Profit_UGX',
    'Waste_Rate',
    # 'Profit_Margin'
]]

# -----------------------------
# 4. SCALE DATA
# -----------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

# -----------------------------
# 5. PCA
# -----------------------------
pca = PCA(n_components=2)
# When creating pca_df, include Academic_Period from your original dataframe
pca_df = pd.DataFrame({
    'PC1': pca_result[:, 0],
    'PC2': pca_result[:, 1],
    'Cluster_Label': df['Cluster_Label'],       # adjust df to your actual dataframe name
    'Academic_Period': df['Academic_Period']     # add this line
})
# -----------------------------
# 6. HDBSCAN CLUSTERING
# -----------------------------
clusterer = hdbscan.HDBSCAN(min_cluster_size=30, min_samples=10)
labels = clusterer.fit_predict(X_scaled)

df['Cluster_Label'] = labels.astype(str)
df.loc[df['Cluster_Label'] == '-1', 'Cluster_Label'] = 'Noise'

# -----------------------------
# 7. CLUSTER STRUCTURE
# -----------------------------
cluster_labels = sorted([c for c in df['Cluster_Label'].unique() if c != 'Noise'])
n_clusters = len(cluster_labels)

pca_df['Cluster_Label'] = df['Cluster_Label']

# -----------------------------
# 8. AGGREGATIONS
# -----------------------------
size_counts = df['Cluster_Label'].value_counts()

df['Academic_Period'] = df.get('Academic_Period', 'Other')
df['Cafeteria_ID'] = df.get('Cafeteria_ID', 'Caf1')

period_pct = pd.crosstab(df['Academic_Period'], df['Cluster_Label'], normalize='columns')
caf_pct = pd.crosstab(df['Cluster_Label'], df['Cafeteria_ID'], normalize='index')

# -----------------------------
# 9. BOXPLOT DATA
# -----------------------------
sold_data = [df[df['Cluster_Label'] == c]['Portions_Sold'] for c in cluster_labels]
waste_data = [df[df['Cluster_Label'] == c]['Waste_Rate'] for c in cluster_labels]
profit_data = [df[df['Cluster_Label'] == c]['Profit_Margin'] for c in cluster_labels]

# -----------------------------
# 10. EXTRA FIELDS
# -----------------------------
df['Daily_Sold'] = df['Portions_Sold']
df['Meal_Entropy'] = np.random.rand(len(df))  # placeholder
df['Is_Noise'] = (df['Cluster_Label'] == 'Noise').astype(int)

hdbscan_results = df.copy()

# -----------------------------
# 11. CHECK OUTPUT
# -----------------------------
print("✅ Variables ready")
print(f"Clusters found: {n_clusters}")
print("\nCluster Distribution:")
print(df['Cluster_Label'].value_counts())

NameError: name 'pca_result' is not defined

In [15]:
# =============================================================
# CELL 86 — HDBSCAN FITTING + ALL VARIABLE SETUP
# Tested against: makerere_Cafeteria_synthetic.csv
# Result: 3 clusters, 11.2% noise  ✅
# =============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.stats import entropy as scipy_entropy
import hdbscan

# -------------------------------------------------------------
# MAKERERE COLOR PALETTE
# -------------------------------------------------------------
MAKERERE_COLORS = {
    'primary':   '#1B4F72',
    'secondary': '#F39C12',
    'accent':    '#E74C3C',
    'green':     '#27AE60',
    'purple':    '#8E44AD',
    'teal':      '#16A085',
    'noise':     '#BDC3C7',
}

CLUSTER_PALETTE = [
    MAKERERE_COLORS['primary'],
    MAKERERE_COLORS['secondary'],
    MAKERERE_COLORS['accent'],
    MAKERERE_COLORS['green'],
    MAKERERE_COLORS['purple'],
    MAKERERE_COLORS['teal'],
    '#2E86AB',
    '#A23B72',
    '#F18F01',
    '#C73E1D',
    '#3B1F2B',
    '#44BBA4',
]

# -------------------------------------------------------------
# 1. LOAD & CLEAN RAW DATA
# -------------------------------------------------------------
df = pd.read_csv('makerere_Cafeteria_synthetic.csv')   # ← adjust path if needed

# Fix currency columns stored as strings e.g. "918,000"
for col in ['Price_UGX', 'Revenue_UGX', 'Ingredient_Cost_UGX', 'Waste_Cost_UGX', 'Gross_Profit_UGX']:
    df[col] = df[col].astype(str).str.replace(',', '').astype(float)

df['Date']       = pd.to_datetime(df['Date'])
df['Is_Weekend'] = df['Is_Weekend'].astype(int)

# -------------------------------------------------------------
# 2. FEATURE ENGINEERING
# -------------------------------------------------------------

# --- Meal Entropy: how evenly demand is spread across the 6 meals ---
def meal_entropy(grp):
    props = grp['Portions_Sold'] / grp['Portions_Sold'].sum()
    return scipy_entropy(props, base=2)

entropy_df = (df.groupby(['Cafeteria_ID', 'Date'])
                .apply(meal_entropy, include_groups=False)
                .reset_index(name='Meal_Entropy'))

# --- Ingredient Diversity: how many distinct ingredients used that day ---
ING_COLS = [
    'Posho_Flour_kg', 'Beans_kg', 'Cooking_Oil_L', 'Matooke_kg',
    'Groundnuts_kg', 'Rice_kg', 'Chicken_kg', 'Offal_kg',
    'Onions_kg', 'Irish_Potatoes_kg', 'Eggs_units',
    'Wheat_Flour_kg', 'Cabbage_kg', 'Tomatoes_kg',
]
ing_div = (df.groupby(['Cafeteria_ID', 'Date'])[ING_COLS]
             .sum()
             .apply(lambda r: (r > 0).sum(), axis=1)
             .reset_index(name='Ingredient_Diversity'))

df['Cost_Per_Portion'] = df['Ingredient_Cost_UGX'] / df['Portions_Sold'].replace(0, np.nan)

# --- Daily aggregate (one row per cafeteria per day) ---
daily = df.groupby(['Cafeteria_ID', 'Date', 'Academic_Period', 'Is_Weekend']).agg(
    Daily_Sold            = ('Portions_Sold',       'sum'),
    Daily_Prepared        = ('Portions_Prepared',   'sum'),
    Daily_Revenue         = ('Revenue_UGX',         'sum'),
    Daily_Ingredient_Cost = ('Ingredient_Cost_UGX', 'sum'),
    Daily_Waste_Portions  = ('Waste_Portions',      'sum'),
    Daily_Gross_Profit    = ('Gross_Profit_UGX',    'sum'),
    Avg_Price_UGX         = ('Price_UGX',           'mean'),
    Avg_Cost_Per_Portion  = ('Cost_Per_Portion',    'mean'),
).reset_index()

# --- Derived ratios ---
daily['Waste_Rate']          = daily['Daily_Waste_Portions'] / daily['Daily_Prepared']
daily['Gross_Profit_Margin'] = daily['Daily_Gross_Profit']   / daily['Daily_Revenue']
daily['Sell_Through_Rate']   = daily['Daily_Sold']           / daily['Daily_Prepared']
daily['Revenue_Per_Portion'] = daily['Daily_Revenue']        / daily['Daily_Sold'].replace(0, np.nan)

daily = daily.merge(entropy_df, on=['Cafeteria_ID', 'Date'])
daily = daily.merge(ing_div,    on=['Cafeteria_ID', 'Date'])

df = daily.copy()

# -------------------------------------------------------------
# 3. FEATURE MATRIX FOR HDBSCAN
# -------------------------------------------------------------
FEATURE_COLS = [
    # DEMAND
    'Daily_Sold',            # total portions sold — separates high/low traffic days
    'Sell_Through_Rate',     # sold / prepared — operational efficiency

    # WASTE
    'Waste_Rate',            # waste / prepared — food loss signal

    # FINANCIALS
    'Gross_Profit_Margin',   # profit / revenue — financial health
    'Revenue_Per_Portion',   # avg revenue per portion sold — pricing signal
    'Avg_Cost_Per_Portion',  # avg ingredient cost per portion — cost structure

    # MENU BEHAVIOUR
    'Meal_Entropy',          # evenness of demand across 6 meals
    'Ingredient_Diversity',  # number of distinct ingredients used that day

    # CONTEXT
    'Is_Weekend',            # 0/1 — weekend vs weekday demand pattern
    'Avg_Price_UGX',         # average menu price that day
]

df_clean = df[FEATURE_COLS + ['Academic_Period', 'Cafeteria_ID']].dropna().copy()
print(f"Rows after dropna: {len(df_clean)}")

X        = df_clean[FEATURE_COLS].values
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

# -------------------------------------------------------------
# 4. PCA — 2D projection for visualization only
# -------------------------------------------------------------
pca        = PCA(n_components=2, random_state=42)
pca_result = pca.fit_transform(X_scaled)
print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.2%}")

# -------------------------------------------------------------
# 5. HDBSCAN CLUSTERING
# -------------------------------------------------------------
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=150,          # raise → fewer clusters | lower → more clusters
    min_samples=30,                # raise → stricter/denser clusters
    cluster_selection_epsilon=0.3, # merge clusters within this distance
    cluster_selection_method='eom',
    metric='euclidean',
    prediction_data=True,
)
labels     = clusterer.fit_predict(X_scaled)
raw_labels = labels.copy()
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

print(f"\nClusters found : {n_clusters}")
print(f"Noise points   : {(labels == -1).sum()}  ({(labels == -1).mean():.1%} of data)")

# -------------------------------------------------------------
# 6. BUILD hdbscan_results
# -------------------------------------------------------------
hdbscan_results = df_clean.copy().reset_index(drop=True)
hdbscan_results['Raw_Label']     = raw_labels
hdbscan_results['Is_Noise']      = (raw_labels == -1).astype(int)
hdbscan_results['Cluster_Label'] = hdbscan_results['Raw_Label'].apply(
    lambda x: 'Noise' if x == -1 else int(x)
)

print("\nCluster Distribution:")
print(hdbscan_results['Cluster_Label'].value_counts())

# -------------------------------------------------------------
# 7. BUILD pca_df  (Academic_Period included — needed for plot 2)
# -------------------------------------------------------------
pca_df = pd.DataFrame({
    'PC1':             pca_result[:, 0],
    'PC2':             pca_result[:, 1],
    'Cluster_Label':   hdbscan_results['Cluster_Label'].values,
    'Academic_Period': hdbscan_results['Academic_Period'].values,
})

# -------------------------------------------------------------
# 8. COLOR MAPS
# -------------------------------------------------------------
unique_labels = sorted([l for l in hdbscan_results['Cluster_Label'].unique() if l != 'Noise'])
color_map     = {lbl: CLUSTER_PALETTE[i % len(CLUSTER_PALETTE)] for i, lbl in enumerate(unique_labels)}
color_map['Noise'] = MAKERERE_COLORS['noise']
sold_colors   = [CLUSTER_PALETTE[i % len(CLUSTER_PALETTE)] for i in range(n_clusters)]

period_colors = {
    'Sem1_Teaching': '#1B4F72',
    'Sem1_Exams':    '#F39C12',
    'Sem_Break':     '#27AE60',
    'Sem2_Teaching': '#8E44AD',
    'Sem2_Exams':    '#E74C3C',
    'Xmas_Break':    '#16A085',
    'Other':         '#BDC3C7',
}

bar_colors = [color_map.get(l, '#999999') for l in hdbscan_results['Cluster_Label'].value_counts().index]

# -------------------------------------------------------------
# 9. CLUSTER SIZE COUNTS
# -------------------------------------------------------------
size_counts = hdbscan_results['Cluster_Label'].value_counts()

# -------------------------------------------------------------
# 10. BOX PLOT DATA — one array per cluster
# -------------------------------------------------------------
sold_data   = [hdbscan_results.loc[hdbscan_results['Cluster_Label'] == i, 'Daily_Sold'].values
               for i in range(n_clusters)]
waste_data  = [hdbscan_results.loc[hdbscan_results['Cluster_Label'] == i, 'Waste_Rate'].values
               for i in range(n_clusters)]
profit_data = [hdbscan_results.loc[hdbscan_results['Cluster_Label'] == i, 'Gross_Profit_Margin'].values
               for i in range(n_clusters)]

# -------------------------------------------------------------
# 11. PERIOD & CAFETERIA PERCENTAGE TABLES
# -------------------------------------------------------------
period_pct = (
    hdbscan_results[hdbscan_results['Cluster_Label'] != 'Noise']
    .groupby(['Cluster_Label', 'Academic_Period'])
    .size()
    .unstack(fill_value=0)
    .apply(lambda r: r / r.sum(), axis=1)
)

caf_pct = (
    hdbscan_results[hdbscan_results['Cluster_Label'] != 'Noise']
    .groupby(['Cluster_Label', 'Cafeteria_ID'])
    .size()
    .unstack(fill_value=0)
    .apply(lambda r: r / r.sum(), axis=1)
)

print("\n✅ All variables ready for the illustration cell.")
print(f"   n_clusters     = {n_clusters}")
print(f"   pca_df shape   = {pca_df.shape}")
print(f"   pca_df columns = {pca_df.columns.tolist()}")

Rows after dropna: 9800
PCA explained variance: 54.21%

Clusters found : 3
Noise points   : 1102  (11.2% of data)

Cluster Distribution:
Cluster_Label
1        3844
0        2725
2        2129
Noise    1102
Name: count, dtype: int64

✅ All variables ready for the illustration cell.
   n_clusters     = 3
   pca_df shape   = (9800, 4)
   pca_df columns = ['PC1', 'PC2', 'Cluster_Label', 'Academic_Period']


In [16]:
# =============================================================
# EXTRA CLUSTER DISTRIBUTION DIAGRAMS — 7 PLOTS
# Run AFTER Cell 86 (hdbscan_results, pca_df, etc. must exist)
# OR run standalone — this file is fully self-contained.
# =============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.stats import entropy as scipy_entropy
import hdbscan

plt.rcParams.update({'font.family': 'DejaVu Sans',
                     'axes.spines.top': False,
                     'axes.spines.right': False})

# -------------------------------------------------------------
# 1. LOAD & PREPARE DATA
# -------------------------------------------------------------
df = pd.read_csv('makerere_Cafeteria_synthetic.csv')   # ← adjust path if needed

for col in ['Price_UGX', 'Revenue_UGX', 'Ingredient_Cost_UGX', 'Waste_Cost_UGX', 'Gross_Profit_UGX']:
    df[col] = df[col].astype(str).str.replace(',', '').astype(float)

df['Date']       = pd.to_datetime(df['Date'])
df['Is_Weekend'] = df['Is_Weekend'].astype(int)

# Meal Entropy
def meal_entropy(grp):
    props = grp['Portions_Sold'] / grp['Portions_Sold'].sum()
    return scipy_entropy(props, base=2)

entropy_df = (df.groupby(['Cafeteria_ID', 'Date'])
                .apply(meal_entropy, include_groups=False)
                .reset_index(name='Meal_Entropy'))

# Ingredient Diversity
ING_COLS = ['Posho_Flour_kg', 'Beans_kg', 'Cooking_Oil_L', 'Matooke_kg',
            'Groundnuts_kg', 'Rice_kg', 'Chicken_kg', 'Offal_kg',
            'Onions_kg', 'Irish_Potatoes_kg', 'Eggs_units',
            'Wheat_Flour_kg', 'Cabbage_kg', 'Tomatoes_kg']

ing_div = (df.groupby(['Cafeteria_ID', 'Date'])[ING_COLS]
             .sum()
             .apply(lambda r: (r > 0).sum(), axis=1)
             .reset_index(name='Ingredient_Diversity'))

df['Cost_Per_Portion'] = df['Ingredient_Cost_UGX'] / df['Portions_Sold'].replace(0, np.nan)

daily = df.groupby(['Cafeteria_ID', 'Date', 'Academic_Period', 'Is_Weekend']).agg(
    Daily_Sold            = ('Portions_Sold',       'sum'),
    Daily_Prepared        = ('Portions_Prepared',   'sum'),
    Daily_Revenue         = ('Revenue_UGX',         'sum'),
    Daily_Ingredient_Cost = ('Ingredient_Cost_UGX', 'sum'),
    Daily_Waste_Portions  = ('Waste_Portions',       'sum'),
    Daily_Gross_Profit    = ('Gross_Profit_UGX',    'sum'),
    Avg_Price_UGX         = ('Price_UGX',           'mean'),
    Avg_Cost_Per_Portion  = ('Cost_Per_Portion',    'mean'),
).reset_index()

daily['Waste_Rate']          = daily['Daily_Waste_Portions'] / daily['Daily_Prepared']
daily['Gross_Profit_Margin'] = daily['Daily_Gross_Profit']   / daily['Daily_Revenue']
daily['Sell_Through_Rate']   = daily['Daily_Sold']           / daily['Daily_Prepared']
daily['Revenue_Per_Portion'] = daily['Daily_Revenue']        / daily['Daily_Sold'].replace(0, np.nan)

daily = daily.merge(entropy_df, on=['Cafeteria_ID', 'Date'])
daily = daily.merge(ing_div,    on=['Cafeteria_ID', 'Date'])
df = daily.copy()

# -------------------------------------------------------------
# 2. FEATURE MATRIX & CLUSTERING
# -------------------------------------------------------------
FEATURE_COLS = [
    'Daily_Sold', 'Sell_Through_Rate', 'Waste_Rate',
    'Gross_Profit_Margin', 'Revenue_Per_Portion', 'Avg_Cost_Per_Portion',
    'Meal_Entropy', 'Ingredient_Diversity', 'Is_Weekend', 'Avg_Price_UGX',
]

df_clean = df[FEATURE_COLS + ['Academic_Period', 'Cafeteria_ID', 'Date']].dropna().copy().reset_index(drop=True)

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(df_clean[FEATURE_COLS].values)

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=150,
    min_samples=30,
    cluster_selection_epsilon=0.3,
    cluster_selection_method='eom',
    metric='euclidean',
    prediction_data=True,
)
labels = clusterer.fit_predict(X_scaled)

df_clean['Cluster_Label'] = ['Noise' if x == -1 else int(x) for x in labels]
df_clean['Is_Noise']      = (labels == -1).astype(int)
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

pca        = PCA(n_components=2, random_state=42)
pca_result = pca.fit_transform(X_scaled)
df_clean['PC1'] = pca_result[:, 0]
df_clean['PC2'] = pca_result[:, 1]
df_clean['Date'] = pd.to_datetime(df_clean['Date'])

print(f"Clusters found : {n_clusters}")
print(f"Noise points   : {(labels == -1).sum()} ({(labels == -1).mean():.1%})")
print(df_clean['Cluster_Label'].value_counts())

# -------------------------------------------------------------
# 3. SHARED STYLING
# -------------------------------------------------------------
COLORS         = ['#1B4F72', '#F39C12', '#27AE60']
NOISE_COLOR    = '#BDC3C7'
CLUSTER_LABELS = ['Cluster 0 (Weekend)', 'Cluster 1 (Weekday Mid)', 'Cluster 2 (Weekday High)']

non_noise = df_clean[df_clean['Cluster_Label'] != 'Noise'].copy()
non_noise['Cluster_Label'] = non_noise['Cluster_Label'].astype(int)

def save_fig(fig, fname, caption):
    fig.text(0.5, 0.02, caption, ha='center', fontsize=9, style='italic', color='#555')
    fig.tight_layout(rect=[0, 0.06, 1, 1])
    fig.savefig(fname, dpi=180, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved {fname}')

# =============================================================
# PLOT 1 — Radar / Spider Chart: Cluster Feature Profiles
# =============================================================
radar_features = ['Daily_Sold', 'Sell_Through_Rate', 'Waste_Rate',
                  'Gross_Profit_Margin', 'Revenue_Per_Portion', 'Meal_Entropy']
radar_labels   = ['Daily\nSold', 'Sell-Through\nRate', 'Waste\nRate',
                  'Profit\nMargin', 'Revenue /\nPortion', 'Meal\nEntropy']

means = non_noise.groupby('Cluster_Label')[radar_features].mean()
norm  = (means - means.min()) / (means.max() - means.min())   # normalise 0-1

N      = len(radar_features)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for i, (idx, row) in enumerate(norm.iterrows()):
    vals = row.tolist() + row.tolist()[:1]
    ax.plot(angles, vals, color=COLORS[i], linewidth=2.5, label=CLUSTER_LABELS[i])
    ax.fill(angles, vals, color=COLORS[i], alpha=0.15)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=11, fontweight='bold')
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75])
ax.set_yticklabels(['25%', '50%', '75%'], fontsize=8, color='grey')
ax.grid(color='grey', alpha=0.3)
ax.set_title('Cluster Feature Profiles (Normalised)', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=10)
save_fig(fig, 'extra_01_radar_profiles.png',
         'Each axis shows a normalised feature mean — higher = stronger signal for that cluster.')

# =============================================================
# PLOT 2 — Stacked Bar: Weekend vs Weekday composition
# =============================================================
fig, ax = plt.subplots(figsize=(9, 6))
wk     = non_noise.groupby(['Cluster_Label', 'Is_Weekend']).size().unstack(fill_value=0)
wk.columns = ['Weekday', 'Weekend']
wk_pct = wk.div(wk.sum(axis=1), axis=0) * 100

bars1 = ax.bar(wk_pct.index, wk_pct['Weekday'], color='#1B4F72', label='Weekday', width=0.5)
bars2 = ax.bar(wk_pct.index, wk_pct['Weekend'], bottom=wk_pct['Weekday'], color='#F39C12', label='Weekend', width=0.5)

for bar, val in zip(bars1, wk_pct['Weekday']):
    if val > 5:
        ax.text(bar.get_x() + bar.get_width() / 2, val / 2, f'{val:.0f}%',
                ha='center', va='center', color='white', fontsize=12, fontweight='bold')
for bar, val, bot in zip(bars2, wk_pct['Weekend'], wk_pct['Weekday']):
    if val > 5:
        ax.text(bar.get_x() + bar.get_width() / 2, bot + val / 2, f'{val:.0f}%',
                ha='center', va='center', color='white', fontsize=12, fontweight='bold')

ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['Cluster 0\n(Weekend)', 'Cluster 1\n(Weekday Mid)', 'Cluster 2\n(Weekday High)'], fontsize=11)
ax.set_ylabel('Percentage of Days (%)', fontsize=11)
ax.set_title('Weekday vs Weekend Composition by Cluster', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 110)
ax.grid(alpha=0.2, axis='y')
save_fig(fig, 'extra_02_weekend_composition.png',
         'Cluster 0 is entirely weekend days; Clusters 1 and 2 are entirely weekday days.')

# =============================================================
# PLOT 3 — Violin: Daily Portions Sold distribution
# =============================================================
fig, ax = plt.subplots(figsize=(10, 6))
data_by_cluster = [non_noise[non_noise['Cluster_Label'] == i]['Daily_Sold'].values
                   for i in range(n_clusters)]
parts = ax.violinplot(data_by_cluster, positions=[0, 1, 2],
                      showmedians=True, showextrema=True)

for pc, col in zip(parts['bodies'], COLORS):
    pc.set_facecolor(col)
    pc.set_alpha(0.7)
parts['cmedians'].set_color('white')
parts['cmedians'].set_linewidth(2)
for key in ['cbars', 'cmaxes', 'cmins']:
    parts[key].set_color('#333')

for i, d in enumerate(data_by_cluster):
    ax.text(i, np.median(d) + 15, f'Md={np.median(d):.0f}',
            ha='center', fontsize=9, fontweight='bold', color='#333')

ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['Cluster 0\n(Weekend)', 'Cluster 1\n(Weekday Mid)', 'Cluster 2\n(Weekday High)'], fontsize=11)
ax.set_ylabel('Daily Portions Sold', fontsize=11)
ax.set_title('Distribution of Daily Portions Sold by Cluster', fontsize=14, fontweight='bold')
ax.grid(alpha=0.2, axis='y')
save_fig(fig, 'extra_03_violin_daily_sold.png',
         'Violins show the full distribution; white line marks the median. Cluster 2 has the highest and widest demand.')

# =============================================================
# PLOT 4 — Heatmap: Cluster × Academic Period (%)
# =============================================================
fig, ax = plt.subplots(figsize=(11, 5))
period_order = ['Sem1_Teaching', 'Sem1_Exams', 'Sem_Break',
                'Sem2_Teaching', 'Sem2_Exams', 'Xmas_Break']
ct     = non_noise.groupby(['Cluster_Label', 'Academic_Period']).size().unstack(fill_value=0)
ct     = ct.reindex(columns=[p for p in period_order if p in ct.columns])
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100

sns.heatmap(ct_pct, ax=ax, cmap='YlOrBr', annot=True, fmt='.1f',
            annot_kws={'size': 11, 'weight': 'bold'}, linewidths=0.5,
            cbar_kws={'label': '% of cluster days'})
ax.set_yticklabels(['Cluster 0 (Weekend)', 'Cluster 1 (Weekday Mid)', 'Cluster 2 (Weekday High)'],
                   rotation=0, fontsize=10)
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right', fontsize=10)
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_title('Academic Period Composition by Cluster (%)', fontsize=14, fontweight='bold')
save_fig(fig, 'extra_04_period_heatmap.png',
         'Each cell shows what percentage of that cluster falls in each academic period.')

# =============================================================
# PLOT 5 — Scatter: Waste Rate vs Gross Profit Margin
# =============================================================
fig, ax = plt.subplots(figsize=(10, 7))

noise_df = df_clean[df_clean['Cluster_Label'] == 'Noise']
ax.scatter(noise_df['Waste_Rate'], noise_df['Gross_Profit_Margin'],
           c=NOISE_COLOR, s=10, alpha=0.2, label='Noise', zorder=1)

for i in range(n_clusters):
    sub = non_noise[non_noise['Cluster_Label'] == i]
    ax.scatter(sub['Waste_Rate'], sub['Gross_Profit_Margin'],
               c=COLORS[i], s=20, alpha=0.55, label=CLUSTER_LABELS[i], zorder=2)
    ax.scatter(sub['Waste_Rate'].mean(), sub['Gross_Profit_Margin'].mean(),
               c=COLORS[i], s=220, marker='*', edgecolors='black', linewidths=0.8, zorder=5)

ax.set_xlabel('Waste Rate (waste / prepared)', fontsize=12)
ax.set_ylabel('Gross Profit Margin', fontsize=12)
ax.set_title('Waste Rate vs Gross Profit Margin by Cluster', fontsize=14, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)
save_fig(fig, 'extra_05_waste_vs_margin.png',
         'Stars mark cluster centroids. Lower waste and higher margin indicate more efficient operation.')

# =============================================================
# PLOT 6 — Stacked Bar: Monthly Cluster Membership Over Time
# =============================================================
fig, ax = plt.subplots(figsize=(13, 6))
df_clean['Month'] = df_clean['Date'].dt.to_period('M')
monthly     = df_clean.groupby(['Month', 'Cluster_Label']).size().unstack(fill_value=0)
months_str  = [str(m) for m in monthly.index]
x           = np.arange(len(months_str))

plot_order  = [0, 1, 2, 'Noise']
plot_colors = COLORS + [NOISE_COLOR]
plot_labels = CLUSTER_LABELS + ['Noise']
bottom      = np.zeros(len(x))

for col, col_color, col_label in zip(plot_order, plot_colors, plot_labels):
    if col in monthly.columns:
        vals = monthly[col].values
        ax.bar(x, vals, bottom=bottom, color=col_color,
               label=col_label, alpha=0.85 if col != 'Noise' else 0.5, width=0.85)
        bottom += vals

ax.set_xticks(x[::2])
ax.set_xticklabels(months_str[::2], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Number of Cafeteria-Days', fontsize=11)
ax.set_title('Monthly Cluster Membership Over Time', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
ax.grid(alpha=0.2, axis='y')
save_fig(fig, 'extra_06_monthly_membership.png',
         'Stacked bars show how cluster membership is distributed across each month of the study period.')

# =============================================================
# PLOT 7 — Heatmap: Cafeteria Cluster Membership (%)
# =============================================================
fig, ax = plt.subplots(figsize=(13, 5))
caf_ct  = non_noise.groupby(['Cafeteria_ID', 'Cluster_Label']).size().unstack(fill_value=0)
caf_pct = caf_ct.div(caf_ct.sum(axis=1), axis=0) * 100

sns.heatmap(caf_pct.T, ax=ax, cmap='Blues', annot=True, fmt='.0f',
            annot_kws={'size': 9}, linewidths=0.5,
            cbar_kws={'label': '% of cafeteria days'})
ax.set_yticklabels(['Cluster 0 (Weekend)', 'Cluster 1 (Weekday Mid)', 'Cluster 2 (Weekday High)'],
                   rotation=0, fontsize=10)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
ax.set_xlabel('Cafeteria ID', fontsize=11)
ax.set_ylabel('')
ax.set_title('Cluster Membership per Cafeteria (%)', fontsize=14, fontweight='bold')
save_fig(fig, 'extra_07_cafeteria_membership.png',
         "Each cell = % of that cafeteria's days assigned to that cluster. Dark = dominant cluster.")

# -------------------------------------------------------------
print('\n✅ All 7 extra plots saved:')
for i, name in enumerate([
    'extra_01_radar_profiles.png',
    'extra_02_weekend_composition.png',
    'extra_03_violin_daily_sold.png',
    'extra_04_period_heatmap.png',
    'extra_05_waste_vs_margin.png',
    'extra_06_monthly_membership.png',
    'extra_07_cafeteria_membership.png',
], 1):
    print(f'  {i}. {name}')

Clusters found : 3
Noise points   : 1102 (11.2%)
Cluster_Label
1        3844
0        2725
2        2129
Noise    1102
Name: count, dtype: int64
Saved extra_01_radar_profiles.png
Saved extra_02_weekend_composition.png
Saved extra_03_violin_daily_sold.png
Saved extra_04_period_heatmap.png
Saved extra_05_waste_vs_margin.png
Saved extra_06_monthly_membership.png
Saved extra_07_cafeteria_membership.png

✅ All 7 extra plots saved:
  1. extra_01_radar_profiles.png
  2. extra_02_weekend_composition.png
  3. extra_03_violin_daily_sold.png
  4. extra_04_period_heatmap.png
  5. extra_05_waste_vs_margin.png
  6. extra_06_monthly_membership.png
  7. extra_07_cafeteria_membership.png
